In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Huang 2-Year Plant Breeding Cycle - AlphaSimPy Tutorial

This notebook converts a BRAID breeding program abstraction into a runnable AlphaSimPy workflow.
It follows the tutorial style used in the AlphaSimPy example notebooks, with sections for imports, parameters, simulation setup, cycle execution, and summary outputs.

**Source**: BRAID abstraction provided by the user  
**Package**: AlphaSimPy  
**Program type**: Repeating 2-year phenotypic selection cycle


## Program Overview

The BRAID abstraction describes a repeating 2-year cycle:

1. Start from a wild founder SP population.
2. Generate GP stock.
3. Select a subset of GP stock as crossing parents.
4. Make crosses to create juvenile SPs.
5. Advance juvenile SPs to mature SPs.
6. Phenotype mature SPs and retain the top 10%.
7. Recycle selected mature SPs back into GP stock for the next cycle.

This notebook implements that logic with explicit assumptions where the BRAID diagram used symbolic quantities.

## Assumptions Used to Make the BRAID Program Executable

The BRAID abstraction includes symbolic quantities such as `NumCross` and `nGP per SP`. To create a deterministic AlphaSimPy notebook, the following assumptions are used:

- Founder population size = 1000.
- Each founder contributes a fixed number of GP stock individuals (`gp_per_founder`).
- A fixed number of crossing parents are selected each cycle.
- Crosses are implemented with `randCross` using selected GP stock.
- Juvenile SPs are advanced to mature SPs without changing population size.
- Mature SP selection is based on phenotype with truncation intensity of 10%.
- Recycled GP stock is generated by random crossing among selected mature SPs as a practical AlphaSimPy approximation of spore release / GP regeneration.

If you have more exact biological or operational details, you can edit the parameter cell below.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from AlphaSimPy import runMacs, SimParam, newPop, randCross, setPheno, selectInd, meanG, varG

print("AlphaSimPy notebook for: Huang 2-Year Plant Breeding Cycle")
print("All libraries imported successfully!")


## Global Parameters

Set up genome, trait, and breeding program parameters derived from the BRAID abstraction.

In [ ]:
# Simulation control
n_reps = 1
n_cycles = 2

# Genome and trait architecture from BRAID
n_chr = 1
n_qtl = 100
n_snp = 0
seg_sites = 2000
founder_size = 1000
h2 = 0.3

# Executable assumptions for symbolic BRAID quantities
gp_per_founder = 2          # Diagram note: 2 GPs per Wild SP
cross_parent_fraction = 0.5 # Diagram note: NumCross/2 GPs used as parents
n_crosses = 100             # Assumed executable value for NumCross
n_progeny = 1               # One juvenile SP per cross to match NumCross total
selected_fraction = 0.10    # Diagram note: top 10% retained
recycled_gp_per_selected = 5  # Assumed executable value for nGP per SP

# Genetic scale assumptions
init_mean_g = 0.0
init_var_g = 1.0
var_e = init_var_g * (1 - h2) / h2

print('Simulation Parameters:')
print(f'  Cycles: {n_cycles}')
print(f'  Chromosomes: {n_chr}')
print(f'  QTL per chromosome: {n_qtl}')
print(f'  Founder population size: {founder_size}')
print(f'  Heritability: {h2}')
print(f'  GP per founder: {gp_per_founder}')
print(f'  Crosses per cycle: {n_crosses}')
print(f'  Progeny per cross: {n_progeny}')
print(f'  Selected fraction among mature SPs: {selected_fraction}')
print(f'  Recycled GP per selected mature SP: {recycled_gp_per_selected}')
print(f'  Error variance used for phenotypes: {var_e:.3f}')


## Simulate Founder Haplotypes and Create the Base Population

This section creates the founder haplotypes and initializes the AlphaSimPy simulation parameters.

In [ ]:
# Simulate founder haplotypes
founder_haps = runMacs(nInd=founder_size, nChr=n_chr, segSites=seg_sites)

# Set simulation parameters
SP = SimParam(founder_haps)
SP.addTraitA(nQtlPerChr=n_qtl, mean=init_mean_g, var=init_var_g)
SP.setVarE(h2=h2)

# Create wild founder SP population
wild_sps = newPop(founder_haps, simParam=SP)

print('Base population created successfully!')
print(f'  Wild SP population size: {wild_sps.nInd}')
print(f'  Mean genetic value: {meanG(wild_sps):.4f}')
print(f'  Genetic variance: {varG(wild_sps):.4f}')


## Define the 2-Year Breeding Cycle

The function below implements one full cycle of the Huang breeding program abstraction.

In [ ]:
def run_cycle(parent_source, cycle_id, simParam=SP):
    """Run one executable approximation of the Huang 2-year breeding cycle."""
    
    # 1. Generate GP stock from the current parent source
    gp_stock = randCross(parent_source, nCrosses=parent_source.nInd, nProgeny=gp_per_founder, simParam=simParam)
    
    # 2. Select crossing parents from GP stock based on phenotype
    setPheno(gp_stock, varE=var_e, simParam=simParam)
    n_cross_parents = max(2, int(gp_stock.nInd * cross_parent_fraction))
    cross_parents = selectInd(gp_stock, nInd=n_cross_parents, use='pheno', simParam=simParam)
    
    # 3. Make crosses to create juvenile SPs
    juvenile_sps = randCross(cross_parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=simParam)
    
    # 4. Advance juvenile SPs to mature SPs
    mature_sps = juvenile_sps
    
    # 5. Phenotype and select top mature SPs
    setPheno(mature_sps, varE=var_e, simParam=simParam)
    n_selected = max(2, int(mature_sps.nInd * selected_fraction))
    selected_mature_sps = selectInd(mature_sps, nInd=n_selected, use='pheno', simParam=simParam)
    
    # 6. Recycle selected mature SPs into GP stock for the next cycle
    recycled_gp_stock = randCross(selected_mature_sps, nCrosses=selected_mature_sps.nInd, nProgeny=recycled_gp_per_selected, simParam=simParam)
    
    # 7. Collect summary statistics
    summary = {
        'cycle': cycle_id,
        'wild_or_input_n': parent_source.nInd,
        'gp_stock_n': gp_stock.nInd,
        'cross_parents_n': cross_parents.nInd,
        'juvenile_sps_n': juvenile_sps.nInd,
        'mature_sps_n': mature_sps.nInd,
        'selected_mature_sps_n': selected_mature_sps.nInd,
        'recycled_gp_stock_n': recycled_gp_stock.nInd,
        'gp_meanG': meanG(gp_stock),
        'juvenile_meanG': meanG(juvenile_sps),
        'mature_meanG': meanG(mature_sps),
        'selected_meanG': meanG(selected_mature_sps),
        'selected_varG': varG(selected_mature_sps)
    }
    
    return {
        'gp_stock': gp_stock,
        'cross_parents': cross_parents,
        'juvenile_sps': juvenile_sps,
        'mature_sps': mature_sps,
        'selected_mature_sps': selected_mature_sps,
        'recycled_gp_stock': recycled_gp_stock,
        'summary': summary
    }


## Run the Breeding Program

We now execute the repeating 2-year cycle and store summary statistics for each cycle.

In [ ]:
cycle_results = []
current_parents = wild_sps

for cycle in range(1, n_cycles + 1):
    result = run_cycle(current_parents, cycle_id=cycle, simParam=SP)
    cycle_results.append(result)
    current_parents = result['selected_mature_sps']
    print(f"Cycle {cycle} complete: selected {result['selected_mature_sps'].nInd} mature SPs, meanG = {result['summary']['selected_meanG']:.4f}")


## Summarize Results

Convert cycle summaries into a table for inspection.

In [ ]:
summary_df = pd.DataFrame([x['summary'] for x in cycle_results])
summary_df


## Plot Genetic Trend Across Cycles

The plot below shows the mean genetic value of selected mature SPs across cycles.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(summary_df['cycle'], summary_df['selected_meanG'], marker='o', linewidth=2)
plt.xlabel('Cycle')
plt.ylabel('Mean Genetic Value of Selected Mature SPs')
plt.title('Genetic Trend Across Huang 2-Year Breeding Cycles')
plt.grid(True, alpha=0.3)
plt.show()


## Plot Population Sizes by Stage

This plot helps verify that the executable approximation follows the intended stage progression.

In [ ]:
plt.figure(figsize=(10, 6))
for col in ['gp_stock_n', 'cross_parents_n', 'juvenile_sps_n', 'selected_mature_sps_n', 'recycled_gp_stock_n']:
    plt.plot(summary_df['cycle'], summary_df[col], marker='o', label=col)
plt.xlabel('Cycle')
plt.ylabel('Population Size')
plt.title('Population Sizes Across Breeding Stages')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Interpretation

This notebook provides an executable AlphaSimPy approximation of the BRAID abstraction for the Huang 2-year breeding cycle.

Key points:

- The stage order from the BRAID workflow is preserved.
- Phenotypic selection is applied to mature SPs, with the top 10% retained.
- Recycled material is used to seed the next cycle.
- Symbolic quantities from the original diagram were replaced with explicit assumptions so the notebook can run end-to-end.

You can refine the assumptions in the parameter cell to better match the biological system or the original breeding diagram.